In [6]:
import numpy as np
from pathlib import Path

# Keep the same environment, states, actions, and opponent as Q-learning.
PLATFORM, PORT, LAUNCH_BINARIES = "mac", 50051, False
TRAIN_EPISODES, MAX_STEPS = 300, 600
EVAL_EVERY, QUICK_EVAL_EPISODES, FINAL_EVAL_EPISODES = 50, 30, 60
LEARNING_RATE, GAMMA, GUARD_REWARD = 0.5, 1.0, 0.10
NUM_STATES, NUM_ACTIONS = 12, 6
DISTANCE_BINS = [0.07, 0.15, 0.30]

import footsiesgym
from footsiesgym.footsies.game import constants
ACTION_NAMES = list(constants.FOOTSIES_ACTION_IDS)
ATTACK_INDICES = {ACTION_NAMES.index(name) for name in ("N_ATTACK", "B_ATTACK", "N_SPECIAL", "B_SPECIAL")}
CHECKPOINT_DIR = Path("footsies_checkpoints")


def encode_state(obs):
    distance = int(np.digitize(float(obs[0]), DISTANCE_BINS))
    opp = np.asarray(obs)[-37:]
    if opp[32] > 0.5:
        phase = 2
    else:
        action = int(np.argmax(opp[8:25]))
        if action not in ATTACK_INDICES:
            phase = 0
        else:
            progress = float(opp[25]) / max(float(opp[26]), 1e-6)
            phase = 1 if progress < 0.65 else 2
    return distance * 3 + phase

def guard_health(obs, opponent=False):
    if opponent:
        return int(np.argmax(obs[55:59]))
    return int(np.argmax(obs[5:9]))

def opponent_action(obs, rng):
    d = float(obs[0])
    if d > 0.30:
        actions, probs = [2, 1, 5], [0.80, 0.10, 0.10]
    elif d > 0.15:
        actions, probs = [2, 5, 3, 1], [0.50, 0.25, 0.15, 0.10]
    elif d > 0.07:
        actions, probs = [3, 5, 1, 0], [0.45, 0.20, 0.20, 0.15]
    else:
        actions, probs = [3, 4, 1, 0], [0.55, 0.25, 0.15, 0.05]
    return int(rng.choice(actions, p=probs))


In [8]:
# One row of logits per state; softmax turns each row into action probabilities.
logits = np.zeros((NUM_STATES, NUM_ACTIONS), dtype=np.float64)


def softmax(x):
    weights = np.exp(x - np.max(x))
    return weights / weights.sum()


def choose_action(state, rng, greedy=False):
    probs = softmax(logits[state])
    return int(np.argmax(probs) if greedy else rng.choice(NUM_ACTIONS, p=probs))


def reinforce_update(states, actions, rewards):
    # Reward-to-go: G[t] = r[t] + gamma * G[t + 1].
    returns = np.empty(len(rewards))
    G = 0.0
    for t in reversed(range(len(rewards))):
        G = rewards[t] + GAMMA * G
        returns[t] = G

    # d log pi(a|s) / d logits[s] = one_hot(a) - pi(.|s).
    gradient = np.zeros_like(logits)
    for s, a, G in zip(states, actions, returns):
        gradient[s] -= G * softmax(logits[s])
        gradient[s, a] += G
    logits[:] += LEARNING_RATE * gradient / len(states)


# Local check: a rewarded action should become more probable.
initial = softmax(logits[0])[3]
reinforce_update([0], [3], [1.0])
updated = softmax(logits[0])[3]
assert updated > initial
logits[:] = 0.0
print(f"Sanity check: P(action=3 | state=0): {initial:.3f} -> {updated:.3f}")


Sanity check: P(action=3 | state=0): 0.167 -> 0.248


**REINFORCE, with no Critic.** For an episode, store `(state, action, reward)`, compute each step's return-to-go $G_t$, then move the policy logits in the direction

$$\nabla_\theta J \propto \sum_t G_t\,\nabla_\theta\log\pi_\theta(a_t\mid s_t).$$

Here $\gamma=1$, so $G_t$ is the sum of rewards from step $t$ to the end. Dividing the accumulated update by episode length is a practical step-size normalization. All policy parameters are updated **after** the episode (unlike Q-learning's step-by-step TD updates).

In [10]:
def run_episode(env, mode="policy", seed=0, train=False):
    agent_rng = np.random.default_rng(seed + 100)
    opponent_rng = np.random.default_rng(seed + 200)
    obs, _ = env.reset(seed=seed)
    states, actions, rewards = [], [], []
    game_return, outcome = 0.0, 0

    for step in range(MAX_STEPS):
        state = encode_state(obs["p1"])
        if mode == "random":
            action = int(agent_rng.integers(NUM_ACTIONS))
        elif mode == "hardcoded":
            action = 2 if float(obs["p1"][0]) > 0.20 else 3
        else:
            action = choose_action(state, agent_rng, greedy=False)

        other = opponent_action(obs["p2"], opponent_rng)
        next_obs, step_rewards, terminated, truncated, _ = env.step({"p1": action, "p2": other})
        r = float(step_rewards["p1"])
        done = bool(terminated["p1"] or truncated["p1"])
        game_return += r

        if train:
            self_loss = guard_health(obs["p1"]) - guard_health(next_obs["p1"])
            opp_loss = guard_health(obs["p1"], True) - guard_health(next_obs["p1"], True)
            shaped_reward = r + GUARD_REWARD * (opp_loss - self_loss)
            states.append(state)
            actions.append(action)
            rewards.append(shaped_reward)

        if done:
            if terminated["p1"]:
                outcome = int(np.sign(r))
            break
        obs = next_obs

    if train:
        reinforce_update(states, actions, rewards)
    return game_return, outcome, step + 1


def evaluate(env, mode="policy", episodes=30, start_seed=10000):
    before = logits.copy()
    results = [run_episode(env, mode, seed=start_seed + i) for i in range(episodes)]
    assert np.array_equal(logits, before), "Evaluation updated the policy!"
    returns, outcomes, lengths = np.asarray(results).T
    return {
        "return": float(returns.mean()),
        "win_rate": float(np.mean(outcomes == 1)),
        "loss_rate": float(np.mean(outcomes == -1)),
        "avg_steps": float(lengths.mean()),
    }


# Vanilla policy gradient (REINFORCE impl)

In [13]:
env = footsiesgym.make(
    config={"port": PORT, "headless": True, "max_t": MAX_STEPS, "action_delay": 0},
    platform=PLATFORM,
    launch_binaries=LAUNCH_BINARIES,
)

try:
    assert env.action_space("p1").n == NUM_ACTIONS
    obs, _ = env.reset()
    assert len(obs["p1"]) == 88
    print("Policy logits shape:", logits.shape)

    best_return, best_policy = -float("inf"), logits.copy()
    for ep in range(1, TRAIN_EPISODES + 1):
        run_episode(env, seed=ep, train=True)
        if ep % EVAL_EVERY == 0:
            stats = evaluate(env, episodes=QUICK_EVAL_EPISODES)
            if stats["return"] > best_return:
                best_return, best_policy = stats["return"], logits.copy()
            print(f"Episode {ep:3d} | eval return={stats['return']:+.3f} | win={stats['win_rate']:.1%}")

    logits[:] = best_policy
    CHECKPOINT_DIR.mkdir(exist_ok=True)
    np.save(CHECKPOINT_DIR / "REINFORCE_best.npy", logits)

    print("\nFinal evaluation (same opponent and seeds):")
    for mode in ("random", "hardcoded", "policy"):
        stats = evaluate(env, mode, FINAL_EVAL_EPISODES, start_seed=20000)
        print(f"{mode:9s} | return={stats['return']:+.3f} | "
              f"win={stats['win_rate']:.1%} | loss={stats['loss_rate']:.1%}")
    print("Saved:", CHECKPOINT_DIR / "REINFORCE_best.npy")
finally:
    env.close()


Policy logits shape: (12, 6)
Episode  50 | eval return=+0.000 | win=50.0%
Episode 100 | eval return=+0.133 | win=56.7%
Episode 150 | eval return=+0.267 | win=63.3%
Episode 200 | eval return=+0.000 | win=50.0%
Episode 250 | eval return=+0.000 | win=50.0%
Episode 300 | eval return=+0.133 | win=56.7%

Final evaluation (same opponent and seeds):
random    | return=+0.067 | win=53.3% | loss=46.7%
hardcoded | return=-1.000 | win=0.0% | loss=100.0%
policy    | return=+0.067 | win=53.3% | loss=46.7%
Saved: footsies_checkpoints/REINFORCE_best.npy


**Compare with Q-learning.** Both agents see the same 12 discrete states, choose among the same 6 actions, and play the same scripted opponent. REINFORCE samples from a learned softmax policy; Q-learning uses an $\epsilon$-greedy policy over learned action values. At evaluation time, both use a greedy choice. The reported game return excludes guard shaping, as in the original notebook.

**Limitations.** REINFORCE waits until the episode ends to update and can have high-variance gradients. The original Q-learning uses $\gamma=0.98$ while this version uses $\gamma=1$ to keep undiscounted episodic policy gradients simple, so this is an educational comparison, not a controlled algorithm benchmark. Next experiment: add a learned baseline to reduce variance without adding TD learning.

**Run requirements.** Install `numpy` and `footsies-gym`, and start the same macOS game server on port `50051` used by the Q-learning notebook (or adjust `PLATFORM`, `PORT`, `LAUNCH_BINARIES`).